# Modelagem
Este notebook tem como objetivo:
- Testar diferentes modelos na base que foi processada em feature_engineering.ipynb
- Definir métricas de performance
- Verificar a performance da baseline
- Selecionar um algoritmo de predição
- Otimizar hiperparâmetros do modelo selecionado
- Propor estratégia de definição de metas
- Propor estratégia de produtização

# Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option('display.max_columns', None)

### Leitura dos dados

In [2]:
df_model = pd.read_csv('Data/processed/base_modelagem_features.csv')
df_model['data'] = pd.to_datetime(df_model['data'])
df_model = df_model.sort_values(by=['codigo_filial', 'data'])
df_model.head()

,codigo_filial,data,faturamento,quantidade,quantidade_tickets,faturamento_medio_ticket,meta_n_med,faixa_vida,tipo_estabelecimento,delivery,metragem_area_venda,panvel_clinic,estacionamento,atendimento_24_horas,local_tipo_mm7,local_tipo_mm15,local_tipo_mm30,tipo_estabelecimento_copy_BAIRRO,tipo_estabelecimento_copy_CENTRO,tipo_estabelecimento_copy_MALL,tipo_estabelecimento_copy_SHOPPING,is_weekend,cluster_comportamento,fat_lag_1,fat_lag_7,fat_lag_8,fat_mm_1,fat_mm_2,fat_mm_3,fat_mm_4,fat_mm_7,dia_do_mes,dia_da_semana,semana_do_mes,is_pagamento,sin_dia_semana,cos_dia_semana,sin_dia_mes,cos_dia_mes,is_feriado,vespera_feriado,is_semana_feriado,qtd_lag_7,qtd_mm_7,preco_medio_mm_7,preco_medio_mm_30,tickets_lag_7,tickets_mm_7,tickets_mm_30,vies_meta_mm_7,target
0,1500,2025-01-01,18077.83704,828.0,138.0,130.998819,12732.76,2,CENTRO,0,309.0388,0,0,0,NaN,NaN,NaN,False,True,False,False,0,0,18077.83704,NaN,NaN,18077.83704,18077.83704,18077.83704,18077.837040,18077.837040,1,2,1,0,0.974928,-0.222521,0.201299,0.979530,1,0,1,NaN,828.0,21.833137,21.833137,NaN,138.000000,138.000000,1.419789,18077.83704
1,1500,2025-01-02,18077.83704,828.0,138.0,130.998819,12732.76,2,CENTRO,0,309.0388,0,0,0,19189.083578,19189.083578,19189.083578,False,True,False,False,0,0,18077.83704,NaN,NaN,18077.83704,18077.83704,18077.83704,18077.837040,18077.837040,2,3,1,0,0.433884,-0.900969,0.394356,0.918958,0,0,0,NaN,828.0,21.833137,21.833137,NaN,138.000000,138.000000,1.419789,29631.50298
2,1500,2025-01-03,29631.50298,1107.0,188.0,157.614378,12732.76,2,CENTRO,0,309.0388,0,0,0,20767.440004,20767.440004,20767.440004,False,True,False,False,0,0,29631.50298,NaN,NaN,29631.50298,23854.67001,21929.05902,21929.059020,21929.059020,3,4,1,0,-0.433884,-0.900969,0.571268,0.820763,0,0,0,NaN,921.0,23.477888,23.477888,NaN,154.666667,154.666667,1.722255,14137.64100
3,1500,2025-01-04,14137.64100,597.0,98.0,144.261643,12732.76,2,CENTRO,0,309.0388,0,0,0,24959.255010,24959.255010,24959.255010,False,True,False,False,1,0,14137.64100,NaN,NaN,14137.64100,21884.57199,20615.66034,19981.204515,19981.204515,4,5,1,0,-0.974928,-0.222521,0.724793,0.688967,0,0,0,NaN,840.0,23.528701,23.528701,NaN,140.500000,140.500000,1.569275,18077.83704
4,1500,2025-01-05,18077.83704,828.0,138.0,130.998819,12732.76,2,CENTRO,0,309.0388,0,0,0,24043.300802,24043.300802,24043.300802,False,True,False,False,1,0,18077.83704,NaN,NaN,18077.83704,16107.73902,20615.66034,19981.204515,19600.531020,5,6,1,1,-0.781831,0.623490,0.848644,0.528964,0,0,0,NaN,837.6,23.189588,23.189588,NaN,140.000000,140.000000,1.539378,21558.72465


### Funções

In [3]:
def calcular_metricas_vendas(y_true, y_pred):
    """
    Calcula as métricas.
    """
    # Erro médio em Reais
    mae = mean_absolute_error(y_true, y_pred)
    
    # Erro quadrático (penaliza grandes desvios)
    rmse = root_mean_squared_error(y_true, y_pred)
    
    # WAPE: Erro percentual ponderado
    # Evita que filiais pequenas distorçam a média
    wape = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)
    
    return {
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2),
        'WAPE': round(wape, 4)
    }

def avaliar_performance_detalhada(df, col_target, col_pred):
    """Avalia o erro em multiplas granularidades do negocio."""
    
    metricas_globais = calcular_metricas_vendas(df[col_target], df[col_pred])
    
    metricas_filiais = df.groupby('codigo_filial').apply(
        lambda x: pd.Series(calcular_metricas_vendas(x[col_target], x[col_pred]))
    ).reset_index()
    
    metricas_clusters = df.groupby('cluster_comportamento').apply(
        lambda x: pd.Series(calcular_metricas_vendas(x[col_target], x[col_pred]))
    ).reset_index()
    
    # Exige que a coluna original tipo_estabelecimento ainda exista no df
    metricas_tipo = df.groupby('tipo_estabelecimento').apply(
        lambda x: pd.Series(calcular_metricas_vendas(x[col_target], x[col_pred]))
    ).reset_index()
    
    return metricas_globais, metricas_filiais, metricas_clusters, metricas_tipo

# Testes
Métricas selecionadas:
- MAE: O dinheiro em sua escala mais pura.
- RMSE: Pra esse caso, diz o quanto o modelo errou nos picos que as filiais apresentam normalmente.
- WAPE: O MAE pode entregar R$500, mas é preciso relativizar com o valor real usando porcentagem.
    - Ao contrario do MAPE, o WAPE vai diferenciar um erro de 5% de uma loja grande pro erro de 25% de uma loja pequena. (O primeiro caso é pior)

In [7]:
FEATURES_NUM = ['faturamento','quantidade','quantidade_tickets','faturamento_medio_ticket','meta_n_med','metragem_area_venda','local_tipo_mm7',
                'local_tipo_mm15', 'local_tipo_mm30','fat_lag_1','fat_lag_7','fat_lag_8', 'fat_mm_1', 'fat_mm_2', 'fat_mm_3', 'fat_mm_4', 'fat_mm_7',
                'dia_do_mes', 'dia_da_semana', 'semana_do_mes', 'sin_dia_semana','cos_dia_semana','sin_dia_mes','cos_dia_mes',
                'qtd_lag_7','qtd_mm_7','preco_medio_mm_7','preco_medio_mm_30','tickets_lag_7','tickets_mm_7','tickets_mm_30','vies_meta_mm_7','target']

FEATURES_CAT = ['faixa_vida']

FEATURES_BOOL = ['delivery','panvel_clinic','estacionamento','atendimento_24_horas', 'tipo_estabelecimento_copy_BAIRRO','tipo_estabelecimento_copy_CENTRO','tipo_estabelecimento_copy_MALL',
                 'tipo_estabelecimento_copy_SHOPPING', 'is_weekend', 'is_pagamento', 'is_feriado', 'vespera_feriado', 'is_semana_feriado']

ID_COLS = ['codigo_filial', 'data','cluster_comportamento']

FEATURES_COL = FEATURES_NUM + FEATURES_CAT + FEATURES_BOOL

## Baseline (Usando a Meta como preditor)
- MAE: 7026.29
- RMSE: 10498.1
- WAPE: 26.97%

In [12]:
# Executa a avaliação (garantindo que não temos NaNs no target)
df_valid = df_model.dropna(subset=['target', 'meta_n_med'])
baseline_global, baseline_por_filial, baseline_por_clusters, baseline_por_tipo = avaliar_performance_detalhada(df_valid, 'target', 'meta_n_med')

print(f"Métricas da Baseline (Meta Atual):")
print(baseline_global)
display(baseline_por_clusters)
display(baseline_por_tipo)

Métricas da Baseline (Meta Atual):
{'MAE': 7026.29, 'RMSE': 10498.14, 'WAPE': np.float64(0.2697)}


,cluster_comportamento,MAE,RMSE,WAPE
0,0,6700.48,8781.86,0.3275
1,1,6796.21,9900.36,0.2358
2,2,37581.46,50206.87,0.2787


,tipo_estabelecimento,MAE,RMSE,WAPE
0,BAIRRO,6847.42,10355.93,0.2654
1,CENTRO,6873.69,9283.02,0.2862
2,MALL,5015.14,6671.34,0.3415
3,SHOPPING,10682.39,17317.70,0.2418


## Regressão Linear

In [9]:
features_treino_num = [f for f in FEATURES_NUM if f != 'target']
features_input = features_treino_num + FEATURES_CAT + FEATURES_BOOL

# Criação da base exclusiva para a regressão, removendo NaNs dos inputs e do target
df_reg = df_model.copy()
df_reg = df_reg.dropna(subset=features_input + ['target'])
df_reg = df_reg.sort_values('data')

data_corte = df_reg['data'].max() - pd.Timedelta(days=30)

train = df_reg[df_reg['data'] < data_corte]
test = df_reg[df_reg['data'] >= data_corte]

X_train = train[features_input]
y_train = train['target']

X_test = test[features_input]
y_test = test['target']

preprocessor = ColumnTransformer([
    ('num_bool', 'passthrough', features_treino_num + FEATURES_BOOL),
    ('cat', OneHotEncoder(handle_unknown='ignore'), FEATURES_CAT)
])

model_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

model_lr.fit(X_train, y_train)

# Usamos .loc para evitar o SettingWithCopyWarning do Pandas
test.loc[:, 'pred_lr'] = model_lr.predict(X_test)

res_lr = calcular_metricas_vendas(y_test, test['pred_lr'])

print(f"WAPE Regressão Linear: {res_lr['WAPE'] * 100:.2f}%")

WAPE Regressão Linear: 31.27%
